In [ ]:
import pandas as pd
from IPython.display import display

from constants import ANALYSIS_ROOT

RESULTS_PATH = ANALYSIS_ROOT / "Hao_PBMC/cell_typing.tsv"
METRICS = ["macro_f1", "balanced_accuracy", "accuracy"]


In [ ]:
results = pd.read_csv(RESULTS_PATH, sep="\t")

stats = (
    results.groupby("representation_type")[METRICS]
    .agg(["mean", "std"])
    .mul(100)
)

summary = pd.DataFrame({
    metric: stats[(metric, "mean")].map("{:.1f}".format)
    + " ± "
    + stats[(metric, "std")].map("{:.1f}".format)
    for metric in METRICS
}).loc[stats[("balanced_accuracy", "mean")].sort_values(ascending=False).index]


In [ ]:
METHOD_INFO = {
    "scunveil_emb_2048": ("Embeddings", 2048, 145.2907178401947, "Yes"),
    "scunveil_emb_512": ("Embeddings", 512, 145.2907178401947, "Yes"),
    "scunveil_emb_128": ("Embeddings", 128, 145.2907178401947, "Yes"),
    "scunveil_emb_50": ("Embeddings", 50, 145.2907178401947, "Yes"),
    "scunveil_emb_10": ("Embeddings", 10, 145.2907178401947, "Yes"),
    "harmony_pca_10": ("Harmony + PCA", 10, 18.48307466506958, "No"),
    "harmony_pca_50": ("Harmony + PCA", 50, 33.9518187046051, "No"),
    "harmony_pca_512": ("Harmony + PCA", 512, 246.13643670082092, "No"),
    "scfoundation": ("scFoundation", 3072, 298000.0, "Yes"),
    "scgpt": ("scGPT", 512, 72781.0, "Yes"),
    "scimilarity": ("SCimilarity", 128, 30.0, "Yes"),
    "scvi_emb_10": ("scVI", 10, 5441.0, "No"),
    "scvi_emb_50": ("scVI", 50, 5601.0, "No"),
    "scvi_emb_512": ("scVI", 512, 5934.0, "No"),
    "uce_4l_1280": ("UCE 4-layer", 1280, 61660.0, "Yes"),
    "uce_33l_1280": ("UCE 33-layer", 1280, 464300.0, "Yes"),
    "magic_50_markers": ("MAGIC Imputed Marker RNA", 50, 110.39749574661255, "No"),
    "scunveil_50_markers": ("Imputed Marker RNA", 50, 217.22, "Yes"),
    "scvi_50_markers": ("scVI Imputed Marker RNA", 50, 5862.0, "No"),
    "rna_log1p": ("Raw RNA", 20729, None, ""),
    "rna_marker_log1p": ("Raw Marker RNA", 50, None, ""),

    "scunveil_linear_emb_2048": ("Linear Embeddings", 2048, 125.26307344436646, "Yes"),
}

method_info = pd.DataFrame.from_dict(
    METHOD_INFO,
    orient="index",
    columns=["label", "dimensionality", "seconds", "zero_shot"],
).rename_axis("representation_type")


In [ ]:
def hms(seconds):
    if pd.isna(seconds):
        return ""
    hours, remainder = divmod(round(seconds), 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


table = summary.join(method_info).reset_index()
table["runtime_hms"] = table["seconds"].map(hms)
table["representation_type"] = table["label"]

table = (
    table.rename(columns={
        "representation_type": "Method",
        "dimensionality": "Dim.",
        "macro_f1": "Macro F1 %",
        "balanced_accuracy": "Balanced Accuracy %",
        "accuracy": "Accuracy %",
        "runtime_hms": "CPU Runtime",
        "zero_shot": "Zero-shot",
    })
    [["Method", "Dim.", "Macro F1 %", "Balanced Accuracy %", "Accuracy %", "CPU Runtime", "Zero-shot"]]
)

display(table)
table.to_csv(ANALYSIS_ROOT / "Hao_PBMC/cell_typing_summary.tsv", sep='\t', index=False)


In [ ]:
counts = (
    results["representation_type"]
    .value_counts()
    .rename_axis("representation_type")
    .reset_index(name="n_rows")
)
counts